**Pipeline:** Load feature store → Optuna tuning (optional) → Full training → Save checkpoint

```
source/datasets/<store_name>/
    ├── train.npz
    ├── val.npz
    ├── test.npz
    └── meta.json
```

## 1. 📦 Import & Config

In [1]:
import sys, os
import json
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from dataclasses import asdict

In [2]:
from config import Config, Indicator
from engine import Engine
from models import Diffusion, DiffusionTransformer
from utils import setup_logging, setup_random_seed, load_processed_store, ExperimentManager
from utils.paths import DATASETS_DIR, CHECKPOINTS_DIR

In [3]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
logger = setup_logging()
logger.info("Imports OK")

2026-04-28 14:09:59,822 - utils.setup - INFO - Logger is set up.
2026-04-28 14:09:59,823 - utils.setup - INFO - Imports OK


In [ ]:
cfg = Config(
    # name              = "trial22_w60h252a14rand42",
    # processed_name    = "trial03_w30seqd30tbs30h292inds292a14rand78",
    skip_training     = False,
    skip_optuna       = False,

    skip_evaluation   = False,
    device            = "cuda:1",
    random_seed       = 78,
)

cfg.feature.symbols = [
   # Tech (20)
    "AAPL", "MSFT", "NVDA", "META", "GOOGL", "AMZN", "AMD",
    "TSLA", "ORCL", "CRM", "ADBE", "INTC", "QCOM", "TXN",
    "AMAT", "MU", "LRCX", "KLAC", "SNPS", "CDNS",
    # Financials (15)
    "JPM", "BAC", "GS", "WFC", "MS", "BLK", "AXP", "C",
    "SCHW", "USB", "PNC", "COF", "TFC", "FITB", "MTB",
    # Healthcare (15)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "TMO", "ABT",
    "BMY", "AMGN", "GILD", "ISRG", "ZTS", "REGN", "VRTX", "CI",
    # Consumer Staples + Discretionary (15)
    "KO", "WMT", "MCD", "NKE", "PG", "COST", "TGT", "SBUX",
    "HD", "LOW", "TJX", "CL", "PEP", "PM", "MO",
    # Industrial + Energy (15)
    "CAT", "XOM", "CVX", "NEE", "BA", "HON", "UPS", "LMT",
    "RTX", "DE", "EMR", "ETN", "GE", "SLB", "OXY",
    # Macro/ETF (20)
    "SPY", "TLT", "GLD", "EEM", "QQQ", "IWM", "DIA", "VNQ",
    "USO", "SLV", "HYG", "AGG", "BND", "VEA", "VWO",
    "ARKK", "XLF", "XLE", "XLK", "XLV"
] # N = 30


test_day_trade = 1002 # 501 days
cfg.inference.window_size = 20
cfg.feature.sequence_depth = 20
cfg.train.epochs = 1000
cfg.inference.horizon = test_day_trade
cfg.inference.normalize_window = True
# Set horizon to the maximum possible given the window size and data length (322)

data_shifted = cfg.inference.horizon
cfg.inference.normalize_shift = data_shifted  # Not necessary to be the same as window_size, but it makes sense to use the same value
# ---------------------------------------
cfg.feature.indicator_shift = data_shifted  # Shift the indicators by the window size to avoid look-ahead bias
cfg.inference.test_batch_stride = cfg.inference.window_size

cfg.processed_name = f"trial06a{len(cfg.feature.symbols)}_w{cfg.inference.window_size}seqd{cfg.feature.sequence_depth}h{cfg.inference.horizon}rand{cfg.random_seed}"

cfg.train.use_scheduler = True
cfg.train.scheduler_type = "warmup_cosine"
# cfg.train.scheduler_type = "plateau"

cfg.name = f"{cfg.processed_name}_model-BATmC_scheduler-{cfg.train.scheduler_type}"

# cfg.optuna.n_trials = 10
# cfg.optuna.epochs_per_trial = 5
# cfg.train.epochs = 20

if not cfg.skip_setup_random_seed:
    setup_random_seed(cfg.random_seed)

DEVICE = torch.device(cfg.device)
print(f"Device : {DEVICE}")
print(f"Config : {cfg.name}  |  store: {cfg.processed_name}")
print(f"Horizon: {cfg.inference.horizon}  |  window: {cfg.inference.window_size}")

Device : cuda:1
Config : trial06a30_w20seqd20h1002rand78_model-BATmC_scheduler-warmup_cosine  |  store: trial06a30_w20seqd20h1002rand78
Horizon: 1002  |  window: 20


## 2 · Load Feature Store

In [5]:
parts, datasets, loaders, meta = load_processed_store(
    store_name   = cfg.processed_name,
    saved_dir    = cfg.save_processed_dir,
    batch_size   = cfg.train.batch_size,
    batch_size_test = cfg.train.batch_size_test,
    shuffle_train= True,
)

train_loader = loaders["train"]
val_loader   = loaders["val"]

# shape for model building and sanity check
sample_batch = next(iter(train_loader))
x_sample    = sample_batch["x"]     # [B, A, T, F]
cond_sample = sample_batch["cond"]  # [B, A, T, F_cond]

# Meta info for model building
NUM_ASSETS        = len(meta["assets"])
NUM_CHANNELS      = meta["shapes"]["train"]["x"][2]     # F target  (such as log-returns = 1)
NUM_COND_CHANNELS = meta["shapes"]["train"]["cond"][2]  # F condition (indicators)
SEQ_LENGTH        = meta["window_size"]

print(f"\nShape check — x: {x_sample.shape}  cond: {cond_sample.shape}")
print(f"num_assets={NUM_ASSETS}  num_channels={NUM_CHANNELS}  "
      f"num_cond_channels={NUM_COND_CHANNELS}  seq_len={SEQ_LENGTH}")

✅ Loaded feature store  →  /home/narodom.y@FUSION.LAB/research/experiments/features/trial06a30_w20seqd20h1002rand78
   train: 1051 windows
   val  : 101 windows
   test : 49 windows

Shape check — x: torch.Size([64, 30, 20, 21])  cond: torch.Size([64, 30, 20, 11])
num_assets=30  num_channels=21  num_cond_channels=11  seq_len=20


In [6]:
for batch in train_loader:
    print("\nBatch keys   :", list(batch.keys()))
    print("x            :", batch["x"].shape)
    print("cond         :", batch["cond"].shape)
    print("close_prices        :", batch["close_prices"].shape)
    print("date     :", batch["dates"].shape)
    print("x_prev_mean       :", batch["x_prev_mean"].shape)
    print("x_prev_std        :", batch["x_prev_std"].shape)
    print("x_prev        :", batch["x_prev"].shape)

    print("x_prev_std min :", batch["x_prev_std"].min())
    print("x_prev_std NaN :", batch["x_prev_std"].isnan().any())
    print("x_prev_std inf :", batch["x_prev_std"].isinf().any())
    break


Batch keys   : ['x', 'cond', 'dates', 'close_prices', 'x_prev_mean', 'x_prev_std', 'x_prev']
x            : torch.Size([64, 30, 20, 21])
cond         : torch.Size([64, 30, 20, 11])
close_prices        : torch.Size([64, 30, 20])
date     : torch.Size([64, 20])
x_prev_mean       : torch.Size([64, 30, 21])
x_prev_std        : torch.Size([64, 30, 21])
x_prev        : torch.Size([64, 30, 20, 21])


x_prev_std min : tensor(0.0020)
x_prev_std NaN : tensor(False)
x_prev_std inf : tensor(False)


In [7]:
batch = next(iter(train_loader))
print(batch["x"].isnan().any())   # ต้องได้ False
print(batch["x"].isinf().any())   # ต้องได้ False
print(batch["x_prev_mean"].isnan().any())  # ต้องได้ False

tensor(False)
tensor(False)
tensor(False)


In [8]:
batch["x"].isnan().any()

tensor(False)

## 3 · Model Factory

In [9]:
def build_model(
    d_model        : int   = 128,
    n_heads        : int   = 4,
    n_layers       : int   = 4,
    dim_feedforward: int   = 512,
    dropout        : float = 0.1,
    noise_steps    : int   = 1000,
    beta_start     : float = 1e-4,
    beta_end       : float = 0.02,
) -> Diffusion:
    """
    Build DiffusionTransformer (backbone) wrapped inside Diffusion (DDPM scheduler).
    Input tensor layout: [B, A, T, C]  (channel-first, ตาม forward ของ DiffusionTransformer)
    """
    backbone = DiffusionTransformer(
        num_assets        = NUM_ASSETS,
        num_channels      = NUM_CHANNELS,
        num_cond_channels = NUM_COND_CHANNELS,
        num_layers        = n_layers,
        num_attention_heads = n_heads,
        seq_length        = SEQ_LENGTH,
        d_model           = d_model,
        dim_feedforward   = dim_feedforward,
        dropout           = dropout,
    )

    model = Diffusion(
        model      = backbone,
        timesteps  = noise_steps,
        beta_start = beta_start,
        beta_end   = beta_end,
    )

    return model


def build_optimizer(model: nn.Module, lr: float, weight_decay: float) -> optim.Optimizer:
    return optim.AdamW(
        model.parameters(),
        lr           = lr,
        betas        = cfg.train.betas,
        eps          = cfg.train.eps,
        weight_decay = weight_decay,
    )


# Quick sanity-check: build default model & count params
_m = build_model().to(DEVICE)
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Default model  →  {total_params:,} trainable params")
del _m

input dim x: 420, d_model: 128
Default model  →  1,231,524 trainable params


In [10]:
def objective(trial: optuna.Trial) -> float:
    """Optuna objective — ส่งคืน best val loss ของ trial นั้น"""

    d_model = trial.suggest_categorical("d_model",  cfg.optuna.suggest_d_model)
    ff_mult = trial.suggest_categorical("ff_mult", cfg.optuna.suggest_ff_mult)
    dim_feedforward = d_model * ff_mult
    trial.set_user_attr("dim_feedforward", dim_feedforward)

    # ── Search space ─────────────────────────────────────────────────────────
    hparams = {
        "d_model"        : d_model,
        "ff_mult"        : ff_mult,
        "dim_feedforward": dim_feedforward,
        "n_heads"        : trial.suggest_categorical("n_heads",  cfg.optuna.suggest_n_heads),
        "n_layers": trial.suggest_int("n_layers", cfg.optuna.suggest_n_layers[0], cfg.optuna.suggest_n_layers[-1]),
        "dropout"        : trial.suggest_float(      "dropout",   cfg.optuna.suggest_dropout[0], cfg.optuna.suggest_dropout[1], step=0.05),
        "noise_steps"    : trial.suggest_categorical("timesteps", cfg.optuna.suggest_noise_steps),
        "lr": trial.suggest_float("lr", cfg.optuna.suggest_lr[0], cfg.optuna.suggest_lr[2], log=True),
        "weight_decay": trial.suggest_float("weight_decay",cfg.optuna.suggest_weight_decay[0],cfg.optuna.suggest_weight_decay[-1], log=True),
        "beta_start": trial.suggest_float("beta_start", cfg.optuna.suggest_beta_start[0], cfg.optuna.suggest_beta_start[-1], log=True),
        "beta_end":   trial.suggest_float("beta_end",   cfg.optuna.suggest_beta_end[0], cfg.optuna.suggest_beta_end[-1], log=True),
    }

    # n_heads must divide d_model evenly
    if hparams["d_model"] % hparams["n_heads"] != 0:
        raise optuna.exceptions.TrialPruned()

    # ── Build model & optimizer ───────────────────────────────────────────────
    model = build_model(
        d_model         = hparams["d_model"],
        n_heads         = hparams["n_heads"],
        n_layers        = hparams["n_layers"],
        dim_feedforward = hparams["dim_feedforward"],
        dropout         = hparams["dropout"],
        noise_steps     = hparams["noise_steps"],
        beta_start = hparams["beta_start"],
        beta_end   = hparams["beta_end"],
    ).to(DEVICE)

    optimizer = build_optimizer(model, hparams["lr"], hparams["weight_decay"])
    criterion = cfg.train.get_criterion()

    if cfg.train.use_scheduler:
        scheduler_type = trial.suggest_categorical("scheduler_type", cfg.optuna.suggest_scheduler_type)
        trial_scheduler = cfg.train.get_scheduler(
            optimizer,
            cfg.optuna.epochs_per_trial,
            scheduler_type=scheduler_type,   # ← pass ตรงนี้
        )
    else:
        trial_scheduler = None

    engine = Engine(
        train_loader      = train_loader,
        val_loader        = val_loader,
        model             = model,
        optimizer         = optimizer,
        criterion         = criterion,
        max_grad_norm     = cfg.train.max_grad_norm,
        clip_gradients    = cfg.train.clip_gradients,
        device            = DEVICE,
        checkpoint_dir    = str(CHECKPOINTS_DIR / "optuna_tmp"),
        checkpoint_filename = f"trial_{trial.number}.pt",
        scheduler           = trial_scheduler,
    )

    # ── Short training loop (epochs_per_trial) ────────────────────────────────
    N_TRIAL_EPOCHS = cfg.optuna.epochs_per_trial
    best_val = float("inf")


    for epoch in range(1, N_TRIAL_EPOCHS + 1):
        engine.train(epoch)
        val_loss = engine.validate(epoch)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val:
            best_val = val_loss

    return best_val

In [11]:
best_hparams = {}

if not cfg.skip_optuna:
    sampler = optuna.samplers.TPESampler(seed=cfg.random_seed)
    pruner  = optuna.pruners.HyperbandPruner(
        min_resource    = cfg.optuna.min_resource,
        max_resource    = cfg.optuna.max_resource,
        reduction_factor= cfg.optuna.reduction_factor,
    )

    study = optuna.create_study(
        direction  = "minimize",
        study_name = cfg.name,
        sampler    = sampler,
        pruner     = pruner,
    )

    study.optimize(objective, n_trials=cfg.optuna.n_trials, show_progress_bar=True)

    best_hparams = study.best_params
    print("\n=== Best Hyperparameters ===")
    print(json.dumps(best_hparams, indent=2))

else:
    print("skip_optuna=True")

  0%|          | 0/100 [00:00<?, ?it/s]

input dim x: 420, d_model: 1024
[W 2026-04-28 14:10:30,209] Trial 0 failed with parameters: {'d_model': 1024, 'ff_mult': 4, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.15000000000000002, 'timesteps': 1250, 'lr': 0.0003296348860614667, 'weight_decay': 9.82316819836239e-06, 'beta_start': 3.85120491994776e-05, 'beta_end': 0.04078986125954618} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/narodom.y@FUSION.LAB/.conda/envs/tsgenai/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_2295809/3028977481.py", line 40, in objective
    optimizer = build_optimizer(model, hparams["lr"], hparams["weight_decay"])
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2295809/2564672739.py", line 38, in build_optimizer
    return optim.AdamW(
           ^^^^^^^^^^^^
  File "/home/naro

KeyboardInterrupt: 

## 5 · Full Training

In [ ]:
# ── Select hyperparameters: Optuna best vs config defaults ────────────────────
if cfg.skip_optuna:
    print("skip_optuna=True — use hyperparameters from config")
    TRAIN_HPARAMS = {
        "d_model":         cfg.model.d_model,
        "n_heads":         cfg.model.n_heads,
        "n_layers":        cfg.model.n_layers,
        "dim_feedforward": cfg.model.dim_feedforward,
        "dropout":         cfg.model.dropout,
        "timesteps":       cfg.model.noise_steps,
        "beta_start":      cfg.model.beta_start,
        "beta_end":        cfg.model.beta_end,
        "lr":              cfg.train.lr,
        "weight_decay":    cfg.train.weight_decay,
        "beta_start":      cfg.model.beta_start,
        "beta_end":        cfg.model.beta_end,
        "scheduler_type":  cfg.train.scheduler_type,
    }
else:
    print(f"skip_optuna=False - use best params from Optuna (value={study.best_value:.6f})")
    TRAIN_HPARAMS = study.best_params
    TRAIN_HPARAMS["dim_feedforward"] = TRAIN_HPARAMS["d_model"] * TRAIN_HPARAMS["ff_mult"]


print("TRAIN_HPARAMS:", TRAIN_HPARAMS)

skip_optuna=True — use hyperparameters from config
TRAIN_HPARAMS: {'d_model': 512, 'n_heads': 4, 'n_layers': 6, 'dim_feedforward': 1024, 'dropout': 0.1, 'timesteps': 1750, 'beta_start': 0.00022330845259098812, 'beta_end': 0.04816694985809928, 'lr': 0.0007343610150369279, 'weight_decay': 2.282803889803081e-06, 'scheduler_type': 'warmup_cosine'}


In [ ]:
TRAIN_HPARAMS

{'d_model': 512,
 'n_heads': 4,
 'n_layers': 6,
 'dim_feedforward': 1024,
 'dropout': 0.1,
 'timesteps': 1750,
 'beta_start': 0.00022330845259098812,
 'beta_end': 0.04816694985809928,
 'lr': 0.0007343610150369279,
 'weight_decay': 2.282803889803081e-06,
 'scheduler_type': 'warmup_cosine'}

In [ ]:
# Experiment manager for organizing checkpoints, logs, etc.
exp = ExperimentManager(config=cfg, exp_name=cfg.name)
print(f"Experiment dir: {exp.exp_dir}")

# ── Build final model ─────────────────────────────────────────────────────────
model = build_model(
    d_model         = TRAIN_HPARAMS["d_model"],
    n_heads         = TRAIN_HPARAMS["n_heads"],
    n_layers        = TRAIN_HPARAMS["n_layers"],
    dim_feedforward = TRAIN_HPARAMS["dim_feedforward"],
    dropout         = TRAIN_HPARAMS["dropout"],
    noise_steps     = TRAIN_HPARAMS["timesteps"],
    beta_start      = TRAIN_HPARAMS["beta_start"],
    beta_end        = TRAIN_HPARAMS["beta_end"],
).to(DEVICE)

optimizer = build_optimizer(model, TRAIN_HPARAMS["lr"], TRAIN_HPARAMS["weight_decay"])
criterion = cfg.train.get_criterion()

# ── Scheduler (optional) ──────────────────────────────────────────────────────
scheduler = None
if cfg.train.use_scheduler:
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer,
    #     T_max  = cfg.train.epochs,
    #     eta_min= cfg.train.eta_min,
    # )
    scheduler = cfg.train.get_scheduler(optimizer, cfg.train.epochs)


# ── Engine ────────────────────────────────────────────────────────────────────
engine = Engine(
    train_loader        = train_loader,
    val_loader          = val_loader,
    model               = model,
    optimizer           = optimizer,
    criterion           = criterion,
    scheduler           = scheduler,
    max_grad_norm       = cfg.train.max_grad_norm,
    clip_gradients      = cfg.train.clip_gradients,
    device              = DEVICE,
    checkpoint_dir      = exp.dirs["checkpoints"],
    checkpoint_filename = f"{cfg.name}.pt",
)

# Save model summary
exp.save_model_summary(model)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel ready  →  {total:,} trainable params")

Experiment dir: /home/narodom.y@FUSION.LAB/research/experiments/20260427_152623_trial06a16_w20seqd20h1002rand78_model-BATmC_scheduler-warmup_cosine_baseline
input dim x: 420, d_model: 512


2026-04-27 15:27:11,484 - Engine - INFO - Engine initialized on cuda:1


2026-04-27 15:27:11,484 - Engine - INFO - Engine initialized on cuda:1


2026-04-27 15:27:11,487 - Engine - INFO - Criterion: MSELoss


2026-04-27 15:27:11,487 - Engine - INFO - Criterion: MSELoss
2026-04-27 15:27:11,493 - ExpManager - INFO - Model summary saved — 20,004,260 trainable params out of 20,004,260 total.

Model ready  →  20,004,260 trainable params


In [ ]:
if not cfg.skip_training:
    engine.fit(
        epochs       = cfg.train.epochs,
        is_save_best = True,
        save_every   = cfg.train.save_every_epochs,
    )
    # Save training artifacts (loss curve, grad norm plot, history CSV)
    exp.save_training_results(engine)

    # Save Optuna study artifacts (ถ้า run Optuna)
    if not cfg.skip_optuna:
        exp.save_optuna_study(study)

    print(f"\nTraining complete. Artifacts saved → {exp.exp_dir}")
else:
    print("skip_training=True — โหลด checkpoint แทน")

2026-04-27 15:27:11,507 - Engine - INFO - Starting training for 5000 epochs...


2026-04-27 15:27:11,507 - Engine - INFO - Starting training for 5000 epochs...


Train Ep 1:   0%|          | 0/18 [00:00<?, ?it/s]

2026-04-27 15:27:19,216 - Engine - INFO - Epoch 1 | Val Loss: 0.9685


2026-04-27 15:27:19,216 - Engine - INFO - Epoch 1 | Val Loss: 0.9685


...

2026-04-27 15:27:30,139 - Engine - INFO - Epoch 5 | Val Loss: 0.6463


2026-04-27 15:27:30,763 - Engine - INFO - New best model saved! (Val Loss: 0.6463)


2026-04-27 15:27:30,763 - Engine - INFO - New best model saved! (Val Loss: 0.6463)


Train Ep 6:   0%|          | 0/18 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6 · Load Best Checkpoint & Quick Sanity Check

In [ ]:
engine.load_checkpoint(f"best_{cfg.name}.pt")
print("Best checkpoint loaded.")

/home/narodom.y@FUSION.LAB/research/src/engine/trainer.py:227: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.

2026-04-03 00:01:44,606 - Engine - INFO - Loaded checkpoint: /ho

2026-04-03 00:01:44,606 - Engine - INFO - Loaded checkpoint: /home/narodom.y@FUSION.LAB/research/experiments/20260403_000021_trial21snap1_w30h252a14rand78_optuna/checkpoints/best_trial21snap1_w30h252a14rand78.pt
Best checkpoint loaded.


In [ ]:
# ── Sanity check: simulate 1 batch from test set ───────────────────────────────
test_loader  = loaders["test"]
sample_batch = next(iter(test_loader))

x_test    = sample_batch["x"].to(DEVICE)     # [B, A, T, F]
cond_test = sample_batch["cond"].to(DEVICE)  # [B, A, T, F_cond]

# Engine.simulate expects [B, A, T, F]

test_loader = loaders["test"]

# Shape info from first batch
sample = next(iter(test_loader))
B, A, T, C      = sample["x"].shape
C_cond          = sample["cond"].shape[-1]

print(f"x    shape : {sample['x'].shape}   → [B, A, T, C]")
print(f"cond shape : {sample['cond'].shape} → [B, A, T, C_cond]")
print(f"Assets     : {A}  |  Window : {T}  |  Channels : {C}  |  Cond channels : {C_cond}")

x    shape : torch.Size([1, 14, 30, 31])   → [B, A, T, C]
cond shape : torch.Size([1, 14, 30, 8]) → [B, A, T, C_cond]
Assets     : 14  |  Window : 30  |  Channels : 31  |  Cond channels : 8
